# Day 2 — Vector Databases

---

Day 1.2's naive linear-scan search worked for 8 documents. Real applications need more:

- **Fast search** over hundreds of thousands of vectors (approximate nearest neighbor, not brute force)
- Multiple users searching at once
- Adding / deleting documents on the fly
- Filtering by metadata ("only 2024 documents from customer X")
- Persistence — no manual save/load

Enter **vector databases**. Today we'll look at three:

1. **ChromaDB** — free, local, embedded (our main tool)
2. **Pinecone** — managed cloud service
3. **pgvector** — Postgres extension (SQL + vectors together)

We'll do hands-on with Chroma and talk about when to reach for the others.


## 1. Vector DB vs a naive Python list — what's actually different?

| Feature | Naive numpy search (Day 2) | Vector DB |
|---|---|---|
| Vector search | ✅ (brute force) | ✅ (approximate nearest neighbor) |
| Scales to 100k+ vectors | 🐌 Slow at query time | ✅ Fast |
| Add / delete a single doc | Awkward | ✅ Easy |
| Store metadata alongside vectors | ❌ (you handle it) | ✅ Built-in |
| Filter by metadata | ❌ | ✅ (`where={"customer_id": 42}`) |
| Multi-user / server mode | ❌ | ✅ |
| Persistence | Manual save/load | ✅ Automatic |

**Rule of thumb:** as soon as your dataset grows past a few thousand vectors, or you need metadata / filters / persistence, reach for a vector DB.


## 2. ChromaDB — free, local, dev-friendly

ChromaDB runs **inside your Python process** — no server to install, no cloud account. Perfect for learning and small projects.


In [ ]:
!pip install chromadb sentence-transformers --quiet

In [1]:
import chromadb

# Save the database on disk so it persists across runs.
client = chromadb.PersistentClient(path="./chroma_data")

# A collection is like a "table" of vectors
abc = client.get_or_create_collection(name="my_docs")
print("Collection created:", abc.name)
print("Database stored at:", "./chroma_data")


Collection created: my_docs
Database stored at: ./chroma_data


## 3. Add documents with metadata

Chroma can either **embed for you** (using a default model) or accept vectors you already computed. For clarity, let's provide our own embeddings.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

docs = [
    "Python is a popular programming language.",
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning models are trained on data.",
    "Croissants are a famous French pastry.",
    "FastAPI is a modern Python web framework.",
    "The Louvre museum houses the Mona Lisa.",
]

metadatas = [
    {"topic": "programming", "year": 2024},
    {"topic": "travel",      "year": 2023},
    {"topic": "programming", "year": 2024},
    {"topic": "food",        "year": 2022},
    {"topic": "programming", "year": 2025},
    {"topic": "travel",      "year": 2023},
]

vectors = model.encode(docs).tolist()   # Chroma wants lists, not numpy arrays
ids = [f"doc_{i}" for i in range(len(docs))]

abc.add(documents=docs, embeddings=vectors, metadatas=metadatas, ids=ids)
print(f"Added {abc.count()} docs")


Added 6 docs


**What we gave Chroma:**
- `documents` — the original text (Chroma stores it for you)
- `embeddings` — the vectors
- `metadatas` — any dict per document (used for filtering later)
- `ids` — unique string IDs so you can update/delete specific docs

Every field except `embeddings` is optional, but you'll want all four in a real app.


## 4. Search Chroma


In [5]:
query_vec = model.encode(["what is the modern web development framework in python?"]).tolist()

results = abc.query(query_embeddings=query_vec, n_results=3)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"  dist={dist:.3f}  ({meta['topic']}, {meta['year']})  {doc}")


  dist=0.615  (programming, 2025)  FastAPI is a modern Python web framework.
  dist=0.720  (programming, 2024)  Python is a popular programming language.
  dist=1.905  (programming, 2024)  Machine learning models are trained on data.


**Note:** Chroma returns **distance**, not similarity — smaller is better (0 = identical). If you want similarity, use `1 - distance`.


## 5. The killer feature — metadata filtering

The whole point of a vector DB is combining semantic search with structured filters. Let's search only in 2024 documents:


In [7]:
results = abc.query(
    query_embeddings=query_vec,
    n_results=3,
    where={"year": 2024}   # only docs where year == 2024
)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  ({meta['year']})  {doc}")


  (2024)  Python is a popular programming language.
  (2024)  Machine learning models are trained on data.


Filters can be more expressive:

```python
where={"year": {"$gte": 2023}}                        # year >= 2023
where={"topic": {"$in": ["food", "travel"]}}          # topic in list
where={"$and": [{"year": 2024}, {"topic": "programming"}]}  # combine
```

In a real app, metadata is everything: `customer_id`, `document_source`, `access_level`, `language`. You'll use `where` constantly.


## 6. Persistent Chroma (survives restarts)

The in-memory client we've been using disappears when Python exits. For real projects use `PersistentClient`:

```python
client = chromadb.PersistentClient(path="./chroma_data")
collection = client.get_or_create_collection(name="my_docs")
```

Everything you add stays in `./chroma_data/`. That's it — no separate server, no config files.


## 7. Pinecone — the managed cloud option

**Pinecone** is a paid, hosted vector database. You don't install anything — you sign up, get an API key, and it just works over HTTPS. Ideal when:

- You have many users
- You don't want to manage infrastructure
- Your dataset is too big for one machine

Free tier gives you one small index — enough to learn on. The API looks like this (illustrative):

```python
from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_3fpvUW_BpLLGmyJGKyJVysR59hm8nftHStFEwVPCSTi11sDdJp4jfBJQLd7DYCzdBoArSj")
pc.create_index(name="notes", dimension=384, metric="cosine",
                spec={"serverless": {"cloud": "aws", "region": "us-east-1"}})

index = pc.Index("testdb321")
index.upsert(vectors=[("doc_1", vector, {"topic": "programming"})])
index.query(vector=query_vec, top_k=3, filter={"topic": "programming"})
```

The mental model is identical to Chroma — collections, ids, vectors, metadata, filters. Only the syntax changes. **If you learn Chroma well, you can switch to Pinecone in an afternoon.**


## 8. pgvector — Postgres + vectors together

If you already use Postgres for your app, **pgvector** lets you store vectors right in your existing database. Great when:

- You want vectors + regular relational data in the same query
- You already have Postgres in production
- You'd rather not run a separate vector DB service

```sql
CREATE EXTENSION vector;
CREATE TABLE docs (id serial, content text, embedding vector(384));

SELECT content FROM docs
ORDER BY embedding <=> '[0.021, -0.117, ...]'  -- cosine distance operator
LIMIT 5;
```

You SQL-search vectors alongside your users, orders, whatever. Very popular in 2025–2026 production apps.


## 9. Which one should you pick?

| Situation | Pick |
|---|---|
| Learning, prototype, side project | **ChromaDB** (local) |
| Live app, don't want to manage servers | **Pinecone** (or Chroma Cloud) |
| Already using Postgres, want one DB | **pgvector** |

For this course we'll stick with **Chroma** — the concepts transfer everywhere.


## Recap

- Vector DBs = fast ANN search + metadata + filtering + persistence + multi-user, out of the box.
- **Chroma** is the easiest to start with — runs in-process, saves to a folder.
- Every doc gets a **vector, an id, and a metadata dict**. Use metadata to filter results.
- **Pinecone** is the same idea, hosted in the cloud. **pgvector** puts vectors inside Postgres.
- **Next class:** chunking (splitting long documents), metadata filtering in depth, and hybrid search (keyword + semantic).
